## 1. Setup

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path
import time
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.paths import MANUAL_PATH, PROCESSED_PATH
from src.macro_sentiment import (
    DEFAULT_MODEL,
    classify_speech_stance,
    extract_speech_text,
    fetch_ecb_speeches,
    fetch_fed_speeches,
    fetch_policy_rates,
    rolling_stance,
    stance_summary,
)

PROJECT_ROOT = Path("..")
DATA_PATH = PROCESSED_PATH

## 2. Fetch Speeches

- **Fed:** RSS feed, only the latest ~15 speeches; the text is loaded from each speech page.
- **ECB:** official full-text export, updated about **once a month** — the latest few weeks are usually missing. Slide decks have no text in the export and are skipped (listed in a warning).

Classifications are cached (section 3), so history accumulates across runs: it starts with the speeches available at the first run. A backfill of older speeches is not done yet.

In [ ]:
N_FED_SPEECHES = 15  # RSS feed only returns the latest ~15 speeches
N_ECB_SPEECHES = 60  # newest speeches in the export; older ones stay in the cache
MODEL = DEFAULT_MODEL  # local Ollama model, recorded per speech in the cache

fed_speeches = fetch_fed_speeches(limit=N_FED_SPEECHES)
ecb_speeches = fetch_ecb_speeches(limit=N_ECB_SPEECHES)

speeches = pd.DataFrame(fed_speeches + ecb_speeches)
speeches["date"] = pd.to_datetime(speeches["date"])
speeches = speeches.sort_values("date", ascending=False).reset_index(drop=True)

print(speeches["central_bank"].value_counts())
speeches[["central_bank", "speaker", "date", "title"]].head(10)

## 3. Load Cache

In [ ]:
RESET_SENTIMENT_CACHE = False

sentiment_cache_path = MANUAL_PATH / "macro_sentiment_cache.xlsx"

cache_cols = [
    "url",
    "central_bank",
    "speaker",
    "title",
    "date",
    "is_monetary_policy",
    "stance_direction",
    "stance_intensity",
    "stance_score",
    "stance_score_llm",
    "confidence",
    "model",
    "key_topics",
    "rationale",
    "text_length",
    "text_preview",
]

if RESET_SENTIMENT_CACHE and sentiment_cache_path.exists():
    sentiment_cache_path.unlink()
    print("Deleted existing sentiment cache.")

if sentiment_cache_path.exists():
    sentiment_cache = pd.read_excel(sentiment_cache_path)
    print(f"Loaded sentiment cache: {len(sentiment_cache)} speeches")
else:
    sentiment_cache = pd.DataFrame(columns=cache_cols)
    print("No sentiment cache found. Starting fresh.")

for col in cache_cols:
    if col not in sentiment_cache.columns:
        sentiment_cache[col] = np.nan

## 4. Classify New Speeches

In [ ]:
already_cached = set(sentiment_cache["url"].dropna())
to_classify = speeches[~speeches["url"].isin(already_cached)]

print(f"Already cached: {len(already_cached)} | New to classify: {len(to_classify)}")

new_rows = []

for _, row in to_classify.iterrows():
    try:
        # the ECB export includes the text, Fed speeches are loaded separately
        if pd.notna(row.get("text")) and row.get("text"):
            text = row["text"]
        else:
            text = extract_speech_text(row["url"])

        if len(text) < 500:
            print(f"Suspiciously short ({len(text)} chars): {row['url']}")

        result = classify_speech_stance(
            text=text,
            speaker=row["speaker"],
            central_bank=row["central_bank"],
            model=MODEL,
        )
        new_rows.append({
            "url": row["url"],
            "central_bank": row["central_bank"],
            "speaker": row["speaker"],
            "title": row["title"],
            "date": row["date"],
            "is_monetary_policy": result.get("is_monetary_policy"),
            "stance_direction": result.get("stance_direction"),
            "stance_intensity": result.get("stance_intensity"),
            "stance_score": result.get("stance_score"),
            "stance_score_llm": result.get("stance_score_llm"),
            "confidence": result.get("confidence"),
            "model": MODEL,
            "key_topics": ", ".join(result.get("key_topics", [])),
            "rationale": result.get("rationale", ""),
            "text_length": len(text),
            "text_preview": text[:300].replace("\n", " "),
        })
    except Exception as e:
        print(f"Failed: {row['url']} ({e})")

    time.sleep(1)

new_rows_df = pd.DataFrame(new_rows)
new_rows_df.head(20)

In [ ]:
if not new_rows_df.empty:
    print(pd.crosstab(new_rows_df["stance_direction"], new_rows_df["stance_intensity"]))
    print(new_rows_df[["stance_score", "stance_score_llm"]].corr())
else:
    print("No new speeches in this run.")

## 5. Update & Save Cache

In [ ]:
sentiment_cache = pd.concat([sentiment_cache, new_rows_df], ignore_index=True)
sentiment_cache = sentiment_cache.drop_duplicates(subset="url", keep="last")

sentiment_cache.to_excel(sentiment_cache_path, index=False)
print(f"Saved {len(sentiment_cache)} speeches to cache.")
sentiment_cache.head(5)

## 6. Time Series, Rolling Average & Summary

Rolling mean over a 90-day calendar window per bank (with only a few speeches per month, shorter windows mostly show single speeches). The summary uses today as reference date, so a source without recent speeches shows up as stale instead of silently ending earlier.

In [ ]:
ROLLING_WINDOW = "90D"
SUMMARY_DAYS = 90

all_speeches = sentiment_cache.copy()
all_speeches["date"] = pd.to_datetime(all_speeches["date"])

n_total = len(all_speeches)
sentiment = all_speeches[all_speeches["is_monetary_policy"] == True]
sentiment = sentiment.dropna(subset=["stance_score"]).sort_values("date")
print(f"Monetary policy speeches: {len(sentiment)} of {n_total}")

sentiment["stance_rolling"] = rolling_stance(sentiment, ROLLING_WINDOW)

stance_summary_df = stance_summary(
    sentiment, all_speeches, as_of=pd.Timestamp.today().normalize(), days=SUMMARY_DAYS
)
stance_summary_df

## 7. Policy Rates (External Reference)

The LLM stance has no ground truth of its own. The actual policy rates — Fed funds target (upper bound, FRED) and ECB deposit facility rate — show whether the classified direction matches what the central banks actually did (e.g. a hawkish lean before a hike). Best effort: without network access the chart runs without them.

In [ ]:
try:
    policy_rates = fetch_policy_rates(start=f"{sentiment['date'].min() - pd.DateOffset(months=6):%Y-%m-%d}")
except Exception as e:
    print(f"Policy rates not available ({e}) -- continuing without them.")
    policy_rates = pd.DataFrame(columns=["date", "central_bank", "policy_rate"])

policy_rates

## 8. Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax_rate = ax.twinx()
colors = {"Fed": "tab:blue", "ECB": "tab:orange"}

for bank, group in sentiment.groupby("central_bank"):
    ax.plot(group["date"], group["stance_rolling"], marker="o", markersize=3,
            color=colors.get(bank), label=f"{bank} stance ({ROLLING_WINDOW} rolling avg)")

for bank, group in policy_rates.groupby("central_bank"):
    group = group[group["date"] >= sentiment["date"].min()]
    ax_rate.step(group["date"], group["policy_rate"], where="post", linestyle=":",
                 color=colors.get(bank), label=f"{bank} policy rate (%)")

ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.set_title("Central Bank Hawkish/Dovish Stance vs. Policy Rate")
ax.set_ylabel("Stance Score (-1 dovish … +1 hawkish)")
ax_rate.set_ylabel("Policy rate (%)")
lines, labels = ax.get_legend_handles_labels()
lines_rate, labels_rate = ax_rate.get_legend_handles_labels()
ax.legend(lines + lines_rate, labels + labels_rate, loc="upper left")
plt.tight_layout()
plt.show()

## 9. Export

In [ ]:
STOPWORDS = {
    "the", "and", "for", "with", "that", "this", "from", "their", "there",
    "have", "has", "been", "will", "would", "could", "should", "about",
    "into", "over", "more", "most", "other", "such", "than", "then", "they",
    "these", "those", "which", "while", "also", "some", "what", "when",
    "policy", "monetary", "speech", "speaker", "central", "bank", "ecb", "fed",
}


def extract_keywords(text):
    words = re.findall(r"[a-zA-Z]{4,}", str(text).lower())
    return [w for w in words if w not in STOPWORDS]


export_cols = [
    "date", "central_bank", "speaker", "title",
    "stance_score", "stance_rolling", "confidence", "key_topics", "rationale",
]

macro_export = sentiment[[c for c in export_cols if c in sentiment.columns]].copy()
macro_export.to_csv(DATA_PATH / "macro_sentiment_timeseries.csv", index=False)

# keyword frequency: each keyword counts once per speech
topic_rows = []
for _, r in sentiment.iterrows():
    for kw in set(extract_keywords(r["key_topics"])):
        topic_rows.append({"central_bank": r["central_bank"], "keyword": kw})

macro_topics = (
    pd.DataFrame(topic_rows)
    .groupby(["central_bank", "keyword"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values("count", ascending=False)
)
macro_topics.to_csv(DATA_PATH / "macro_sentiment_topics.csv", index=False)

stance_summary_df.to_csv(DATA_PATH / "macro_sentiment_summary.csv", index=False)
policy_rates.to_csv(DATA_PATH / "macro_policy_rates.csv", index=False)

print(f"Exported {len(macro_export)} speeches, {len(macro_topics)} keyword rows, "
      f"{len(policy_rates)} policy rate points.")
macro_topics.head(15)